# Clasificación automática de intención en mensajes de clientes

## Contexto práctico

Una empresa recibe mensajes por chat, correo y formularios de contacto. Cada mensaje necesita ser enviado al área adecuada: logística, facturación, devoluciones, ventas o soporte.

Cuando todos los mensajes se clasifican manualmente, el proceso puede ser lento y producir retrasos. En este ejercicio construiremos un clasificador tradicional de texto que identifique la **intención principal** de cada mensaje.

**Importante:** este notebook no analiza si el cliente está feliz o molesto. Analiza qué necesita el cliente. Por ejemplo, “Mi pedido no llega” puede ser una solicitud de seguimiento, aunque el tono del mensaje sea positivo, negativo o neutral.

## ¿Qué problema se resuelve?

El sistema recibirá un mensaje y devolverá una categoría de intención:

| Intención | Ejemplo | Área sugerida |
|---|---|---|
| seguimiento_pedido | “Quiero saber dónde está mi pedido” | Logística |
| facturacion | “Me cobraron dos veces” | Facturación |
| devolucion | “Quiero regresar el producto” | Devoluciones |
| informacion | “¿Qué métodos de pago aceptan?” | Ventas |
| producto_danado | “El artículo llegó roto” | Soporte |
| cancelacion | “Deseo cancelar mi compra” | Cancelaciones |

La utilidad es automatizar la clasificación inicial de tickets. Un empleado todavía puede revisar los casos difíciles, pero los mensajes sencillos se pueden dirigir automáticamente.

## Objetivos de aprendizaje

- Comprender la diferencia entre sentimiento e intención.
- Trabajar con un dataset incluido en el notebook.
- Convertir texto en características numéricas.
- Utilizar palabras individuales y combinaciones de palabras.
- Entrenar un clasificador `MultinomialNB`.
- Evaluar sus aciertos y errores.
- Probar mensajes nuevos.
- Interpretar los resultados desde una perspectiva de negocio.

El dataset es pequeño y educativo. En un proyecto real se necesitarían muchos más ejemplos y etiquetas revisadas por expertos.

## ¿Qué técnica se utilizará?

Aplicaremos **clasificación supervisada de texto**. Esto significa que el modelo aprende a partir de ejemplos que ya tienen una categoría conocida.

El proceso será:

1. Tenemos mensajes con su intención real.
2. `CountVectorizer` transforma los textos en una matriz de conteos.
3. Los **n-gramas** permiten conservar expresiones de dos palabras, como “cobro duplicado” o “seguir pedido”.
4. `MultinomialNB` aprende qué palabras y expresiones son frecuentes en cada intención.
5. El modelo clasifica mensajes que no había visto.

No se utiliza regresión logística ni análisis de sentimiento en este ejercicio.

## 1. Preparar el entorno

Instalamos las librerías necesarias. `%%capture` oculta mensajes informativos de instalación para que el notebook sea más limpio en Google Colab.

In [ ]:
%%capture
!pip -q install pandas scikit-learn plotly

### Importar librerías

- `pandas` organiza el dataset.
- `plotly` crea gráficas interactivas.
- `train_test_split` separa ejemplos para entrenar y evaluar.
- `CountVectorizer` convierte texto en variables numéricas.
- `MultinomialNB` clasifica textos representados por conteos.
- Las métricas muestran qué tan bien funciona el modelo.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option('display.max_colwidth', 120)
print('Entorno preparado correctamente.')

## 2. Dataset incluido en el notebook

Cada fila representa un mensaje de cliente. La columna `intencion_real` es la respuesta de referencia que usaremos para enseñar y evaluar el modelo.

El modelo no utilizará directamente esa columna para predecir. Primero verá los textos y sus etiquetas durante el entrenamiento; después se evaluará con mensajes separados.

In [ ]:
datos = [
    {'id': 'T001', 'canal': 'chat', 'mensaje': 'Quiero saber dónde está mi pedido', 'intencion_real': 'seguimiento_pedido'},
    {'id': 'T002', 'canal': 'correo', 'mensaje': '¿Cuándo llegará mi paquete?', 'intencion_real': 'seguimiento_pedido'},
    {'id': 'T003', 'canal': 'web', 'mensaje': 'Necesito consultar el estado del envío', 'intencion_real': 'seguimiento_pedido'},
    {'id': 'T004', 'canal': 'chat', 'mensaje': 'Mi pedido todavía no llega', 'intencion_real': 'seguimiento_pedido'},
    {'id': 'T005', 'canal': 'correo', 'mensaje': '¿Me pueden compartir el número de guía?', 'intencion_real': 'seguimiento_pedido'},

    {'id': 'T006', 'canal': 'chat', 'mensaje': 'Me cobraron dos veces el mismo producto', 'intencion_real': 'facturacion'},
    {'id': 'T007', 'canal': 'correo', 'mensaje': 'Necesito una copia de mi factura', 'intencion_real': 'facturacion'},
    {'id': 'T008', 'canal': 'web', 'mensaje': 'El importe de mi recibo es incorrecto', 'intencion_real': 'facturacion'},
    {'id': 'T009', 'canal': 'chat', 'mensaje': 'Quiero aclarar un cargo de mi tarjeta', 'intencion_real': 'facturacion'},
    {'id': 'T010', 'canal': 'correo', 'mensaje': 'Tengo un problema con el cobro', 'intencion_real': 'facturacion'},

    {'id': 'T011', 'canal': 'chat', 'mensaje': 'Quiero regresar el producto', 'intencion_real': 'devolucion'},
    {'id': 'T012', 'canal': 'correo', 'mensaje': '¿Cómo puedo solicitar una devolución?', 'intencion_real': 'devolucion'},
    {'id': 'T013', 'canal': 'web', 'mensaje': 'Deseo devolver mi compra', 'intencion_real': 'devolucion'},
    {'id': 'T014', 'canal': 'chat', 'mensaje': 'Necesito cambiar un artículo que compré', 'intencion_real': 'devolucion'},
    {'id': 'T015', 'canal': 'correo', 'mensaje': 'Quiero conocer el proceso para regresar un pedido', 'intencion_real': 'devolucion'},

    {'id': 'T016', 'canal': 'chat', 'mensaje': '¿Qué métodos de pago aceptan?', 'intencion_real': 'informacion'},
    {'id': 'T017', 'canal': 'correo', 'mensaje': '¿Cuál es el horario de atención?', 'intencion_real': 'informacion'},
    {'id': 'T018', 'canal': 'web', 'mensaje': 'Quisiera conocer las características del producto', 'intencion_real': 'informacion'},
    {'id': 'T019', 'canal': 'chat', 'mensaje': '¿Cuánto cuesta el envío?', 'intencion_real': 'informacion'},
    {'id': 'T020', 'canal': 'correo', 'mensaje': 'Necesito información sobre la garantía', 'intencion_real': 'informacion'},

    {'id': 'T021', 'canal': 'chat', 'mensaje': 'El artículo llegó roto', 'intencion_real': 'producto_danado'},
    {'id': 'T022', 'canal': 'correo', 'mensaje': 'Recibí el producto dañado', 'intencion_real': 'producto_danado'},
    {'id': 'T023', 'canal': 'web', 'mensaje': 'El paquete llegó golpeado y el equipo no funciona', 'intencion_real': 'producto_danado'},
    {'id': 'T024', 'canal': 'chat', 'mensaje': 'Mi artículo tiene una pieza rota', 'intencion_real': 'producto_danado'},
    {'id': 'T025', 'canal': 'correo', 'mensaje': 'El producto llegó con daños', 'intencion_real': 'producto_danado'},

    {'id': 'T026', 'canal': 'chat', 'mensaje': 'Deseo cancelar mi compra', 'intencion_real': 'cancelacion'},
    {'id': 'T027', 'canal': 'correo', 'mensaje': 'Quiero anular el pedido', 'intencion_real': 'cancelacion'},
    {'id': 'T028', 'canal': 'web', 'mensaje': '¿Cómo cancelo la orden?', 'intencion_real': 'cancelacion'},
    {'id': 'T029', 'canal': 'chat', 'mensaje': 'Necesito detener la compra', 'intencion_real': 'cancelacion'},
    {'id': 'T030', 'canal': 'correo', 'mensaje': 'Por favor cancelen mi pedido', 'intencion_real': 'cancelacion'}
]

df = pd.DataFrame(datos)
print(f'Dataset cargado: {len(df)} mensajes y {df.shape[1]} columnas.')
display(df.head(8))

### Interpretación del dataset

Tenemos 30 ejemplos distribuidos en seis intenciones. Cada categoría tiene cinco mensajes, por lo que el dataset está balanceado para fines didácticos.

El campo `canal` no se usará para decidir la intención. Se conserva para observar después si la distribución de mensajes cambia entre chat, correo y web.

In [ ]:
resumen_intenciones = (
    df['intencion_real'].value_counts()
      .rename_axis('intencion')
      .reset_index(name='cantidad')
      .sort_values('intencion')
)
display(resumen_intenciones)

fig = px.bar(
    resumen_intenciones,
    x='intencion', y='cantidad', text='cantidad', color='intencion',
    title='Distribución de mensajes por intención',
    labels={'intencion': 'Intención', 'cantidad': 'Mensajes'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=480, xaxis_tickangle=-25)
fig.show()

## 3. Separar entrenamiento y evaluación

No debemos evaluar el modelo únicamente con los mismos textos que utilizó para aprender. Por eso separamos los datos:

- **Entrenamiento:** 70% de los mensajes. El modelo aprende patrones.
- **Prueba:** 30% de los mensajes. Comprobamos si generaliza a textos que no vio.

`stratify` conserva una proporción semejante de intenciones en ambos grupos. `random_state` hace que el resultado sea reproducible.

In [ ]:
X = df['mensaje']
y = df['intencion_real']

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f'Mensajes para entrenamiento: {len(X_entrenamiento)}')
print(f'Mensajes para prueba: {len(X_prueba)}')
print('\nDistribución de intenciones en prueba:')
display(y_prueba.value_counts().sort_index())

## 4. Convertir los mensajes en características

Los algoritmos tradicionales no trabajan directamente con frases. Necesitamos convertir cada texto en números.

`CountVectorizer` crea una columna por palabra o combinación de palabras. El valor indica cuántas veces aparece en cada mensaje.

Usaremos `ngram_range=(1, 2)`:

- Unigramas: “pedido”, “factura”, “cancelar”.
- Bigramas: “seguir pedido”, “cobro duplicado”, “producto dañado”.

Los bigramas ayudan a conservar un poco de contexto. Por ejemplo, “producto” y “dañado” por separado son útiles, pero “producto dañado” es una señal todavía más clara.

In [ ]:
vectorizador = CountVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 2),
    min_df=1
)

matriz_entrenamiento = vectorizador.fit_transform(X_entrenamiento)
matriz_prueba = vectorizador.transform(X_prueba)

print(f'Características creadas: {len(vectorizador.get_feature_names_out())}')
print(f'Dimensiones de la matriz de entrenamiento: {matriz_entrenamiento.shape}')
print('\nAlgunas características aprendidas:')
print(vectorizador.get_feature_names_out()[:30])

### ¿Cómo interpretar la matriz?

Si la matriz tiene forma `(21, 100)`, significa que hay 21 mensajes de entrenamiento y 100 características diferentes. Cada fila representa un mensaje; cada columna representa una palabra o bigrama.

La matriz suele ser dispersa: la mayoría de las palabras no aparecen en cada mensaje. Esto es normal en minería de texto y permite trabajar con vocabularios grandes.

## 5. Entrenar el clasificador Multinomial Naive Bayes

`MultinomialNB` es un modelo sencillo y eficiente para textos representados mediante conteos. Aprende qué términos aparecen con mayor frecuencia en cada intención.

Por ejemplo, si la palabra “factura” aparece principalmente en mensajes de facturación, el modelo la utilizará como señal para esa categoría. No memoriza una sola regla; combina muchas señales y calcula cuál intención es más probable.

El parámetro `alpha` aplica suavizado. Esto evita problemas cuando una palabra no aparece en una categoría durante el entrenamiento.

In [ ]:
modelo = Pipeline([
    ('vectorizador', vectorizador),
    ('clasificador', MultinomialNB(alpha=1.0))
])

modelo.fit(X_entrenamiento, y_entrenamiento)
print('Modelo entrenado correctamente.')

## 6. Evaluar las predicciones

Ahora el modelo clasificará los mensajes de prueba. Como sus intenciones reales sí son conocidas, podemos comparar la predicción con la respuesta esperada.

- **Accuracy:** porcentaje total de aciertos.
- **Precision:** de los mensajes asignados a una intención, cuántos eran realmente de esa intención.
- **Recall:** de los mensajes que pertenecían a una intención, cuántos logró encontrar el modelo.
- **F1:** equilibrio entre precision y recall.

Con un dataset pequeño, estas métricas pueden cambiar mucho si cambia la división entre entrenamiento y prueba.

In [ ]:
predicciones = modelo.predict(X_prueba)
exactitud = accuracy_score(y_prueba, predicciones)

print(f'Exactitud en datos de prueba: {exactitud:.2%}')
print('\nReporte de clasificación:')
print(classification_report(y_prueba, predicciones, zero_division=0))

resultados_prueba = pd.DataFrame({
    'mensaje': X_prueba.values,
    'intencion_real': y_prueba.values,
    'intencion_predicha': predicciones
})
resultados_prueba['acierto'] = resultados_prueba['intencion_real'] == resultados_prueba['intencion_predicha']
display(resultados_prueba.reset_index(drop=True))

### Interpretación de la evaluación

La exactitud indica cuántos mensajes fueron clasificados correctamente en conjunto. El reporte permite detectar si una intención específica funciona peor que las demás.

No conviene concluir que el modelo es excelente solo porque obtiene una exactitud alta. El dataset es pequeño, las frases son relativamente claras y las categorías están balanceadas. En producción habría que probar con mensajes reales, abreviaturas, errores ortográficos, frases largas y casos ambiguos.

## 7. Matriz de confusión

La matriz de confusión muestra las categorías reales en las filas y las categorías predichas en las columnas.

- La diagonal contiene los aciertos.
- Las celdas fuera de la diagonal contienen errores.
- Una confusión entre `seguimiento_pedido` e `informacion` puede indicar que ambas intenciones comparten palabras como “pedido” o “información”.

In [ ]:
etiquetas = sorted(df['intencion_real'].unique())
matriz = confusion_matrix(y_prueba, predicciones, labels=etiquetas)

fig = px.imshow(
    matriz,
    x=etiquetas,
    y=etiquetas,
    text_auto=True,
    color_continuous_scale='Blues',
    labels={'x': 'Intención predicha', 'y': 'Intención real', 'color': 'Cantidad'},
    title='Matriz de confusión: intención real frente a intención predicha'
)
fig.update_layout(height=600, xaxis_tickangle=-35)
fig.show()

### Cómo interpretar la matriz

Si una celda diagonal tiene valor 2, significa que dos mensajes de esa intención fueron clasificados correctamente. Si una celda fuera de la diagonal tiene valor 1, significa que un mensaje fue enviado a una categoría incorrecta.

En un sistema real, los errores más importantes son aquellos que dirigen un caso urgente al área equivocada. Por ejemplo, confundir un producto dañado con una simple solicitud de información puede retrasar una solución al cliente.

## 8. Revisar el volumen por canal

Esta visualización no evalúa al modelo. Sirve para conocer de dónde llegan los mensajes del dataset.

En un escenario real, este análisis ayuda a responder preguntas como: ¿el chat concentra más cancelaciones?, ¿el correo recibe más solicitudes de facturación?, ¿el formulario web se utiliza principalmente para pedir información?

In [ ]:
volumen_canal = (
    df.groupby(['canal', 'intencion_real'])
      .size()
      .reset_index(name='cantidad')
)

fig = px.bar(
    volumen_canal,
    x='canal', y='cantidad', color='intencion_real',
    barmode='group', text='cantidad',
    title='Distribución de intenciones por canal',
    labels={'canal': 'Canal', 'cantidad': 'Mensajes', 'intencion_real': 'Intención'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='outside')
fig.update_layout(height=520)
fig.show()

## 9. Clasificar mensajes nuevos

Este es el uso práctico del modelo: recibir mensajes que no tienen etiqueta y asignarles una intención.

También mostraremos las probabilidades estimadas. No deben interpretarse como certeza absoluta; sirven para identificar casos con menor confianza y enviarlos a revisión humana.

In [ ]:
mensajes_nuevos = [
    '¿Dónde puedo ver el rastreo de mi paquete?',
    'La factura tiene un cargo que no reconozco',
    'Quiero regresar el artículo porque no lo necesito',
    '¿Puedo pagar con tarjeta de crédito?',
    'Mi equipo llegó roto y golpeado',
    'Necesito cancelar la orden de hoy'
]

predicciones_nuevas = modelo.predict(mensajes_nuevos)
probabilidades_nuevas = modelo.predict_proba(mensajes_nuevos)
confianzas = probabilidades_nuevas.max(axis=1)

tabla_nuevos = pd.DataFrame({
    'mensaje': mensajes_nuevos,
    'intencion_predicha': predicciones_nuevas,
    'confianza_aproximada': confianzas.round(3)
})
display(tabla_nuevos)

### Interpretación de los mensajes nuevos

Cada mensaje recibe una categoría que puede utilizarse para enrutar el ticket. Por ejemplo:

- `seguimiento_pedido` puede enviarse a logística.
- `facturacion` puede enviarse al área administrativa.
- `producto_danado` puede recibir prioridad de soporte.

Si la confianza es baja o las dos mejores categorías tienen valores muy parecidos, conviene solicitar revisión humana. Una automatización responsable no debe ocultar la incertidumbre.

## 10. Ver las palabras más asociadas a cada intención

Una ventaja de este enfoque tradicional es que podemos inspeccionar el vocabulario utilizado por el clasificador. Las diferencias entre los pesos logarítmicos ayudan a entender qué términos favorecen una categoría.

Esta inspección no es una explicación perfecta del modelo, pero sirve para detectar vocabulario útil, términos ambiguos y posibles errores de etiquetado.

In [ ]:
vectorizador_entrenado = modelo.named_steps['vectorizador']
clasificador_entrenado = modelo.named_steps['clasificador']
caracteristicas = vectorizador_entrenado.get_feature_names_out()

filas_vocabulario = []
for indice_clase, clase in enumerate(clasificador_entrenado.classes_):
    mejores_indices = clasificador_entrenado.feature_log_prob_[indice_clase].argsort()[-8:][::-1]
    for indice_caracteristica in mejores_indices:
        filas_vocabulario.append({
            'intencion': clase,
            'termino_importante': caracteristicas[indice_caracteristica],
            'nivel_relativo': round(clasificador_entrenado.feature_log_prob_[indice_clase, indice_caracteristica], 3)
        })

vocabulario_por_intencion = pd.DataFrame(filas_vocabulario)
display(vocabulario_por_intencion)

fig = px.bar(
    vocabulario_por_intencion,
    x='nivel_relativo', y='termino_importante', color='intencion',
    facet_col='intencion', facet_col_wrap=2, orientation='h',
    title='Términos frecuentes aprendidos por intención',
    labels={'nivel_relativo': 'Peso logarítmico relativo', 'termino_importante': 'Término'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(height=900, showlegend=False)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))
fig.show()

### Interpretación del vocabulario

Los términos deben interpretarse junto con su intención. Palabras como “factura”, “cobro” o “cargo” aportan señales para `facturacion`; palabras como “cancelar”, “anular” o “detener” aportan señales para `cancelacion`.

Si aparece un término poco relacionado, puede ser una señal de que el dataset es demasiado pequeño o de que las etiquetas necesitan revisión. Esta revisión ayuda a mejorar el diccionario de ejemplos y a incorporar nuevas expresiones.

## 11. Conclusiones generales

1. La clasificación de intención responde a la pregunta **“¿qué necesita el cliente?”**, mientras que el análisis de sentimiento responde a **“¿cómo se siente?”**.
2. `CountVectorizer` permite convertir mensajes en variables numéricas usando palabras y bigramas.
3. Los n-gramas ayudan a conservar expresiones importantes como “cobro duplicado” o “producto dañado”.
4. `MultinomialNB` es una alternativa sencilla, rápida y apropiada para una primera solución de clasificación de textos.
5. La matriz de confusión permite detectar qué intenciones se confunden.
6. Las predicciones nuevas pueden utilizarse para enrutar tickets automáticamente.
7. La confianza baja debe activar una revisión humana, especialmente en casos de devolución, facturación o productos dañados.
8. El resultado educativo no debe generalizarse directamente a producción: se necesitan más datos, más variedad lingüística y validación con usuarios reales.

### Conclusión ejecutiva

El ejercicio muestra cómo una empresa puede transformar mensajes libres en una clasificación operativa. Con una solución sencilla es posible reducir trabajo manual y acelerar la asignación de tickets. El beneficio principal no es solo clasificar, sino convertir texto no estructurado en una acción concreta: enviar cada caso al área que puede resolverlo.

## 12. Posibles mejoras

Para evolucionar este ejercicio se podrían:

- incorporar cientos o miles de mensajes reales;
- agregar errores ortográficos, abreviaturas y lenguaje informal;
- comparar `MultinomialNB` con SVM o regresión logística;
- probar TF-IDF y embeddings como representaciones alternativas;
- crear una categoría “requiere revisión humana”;
- medir el costo de cada tipo de error;
- actualizar periódicamente el modelo con nuevos tickets etiquetados.